#####Setups

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

base_path = "s3://l1l2bucket/data/raw/giottus"

catalog = "crypto_lakehouse"
bronze_schema = "bronze"


In [ ]:
display(dbutils.fs.ls(base_path))

####Giottus balances

In [ ]:
spot_balances_schema = StructType([
    StructField("asset", StringType(), True),
    StructField("free", StringType(), True),
    StructField("locked", StringType(), True),
    StructField("lockedFd", StringType(), True),
    StructField("lockedOtc", StringType(), True),
    StructField("lockedStaking", StringType(), True)
])

In [ ]:
spot_balances_raw_df = (
    spark.read
    .schema(spot_balances_schema)
    .option("multiLine", "true")
    .option("recursiveFileLookup", True)
    .json(f"{base_path}/balances")
)

spot_balances_df = (
    spot_balances_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Giottus"))
    .withColumn("source_type", lit("Spot_Balance"))
    .withColumn("ingested_at", current_timestamp())
)

print("Spot balance rows:", spot_balances_df.count())

display(
    spot_balances_df.orderBy(
        col("asset").desc()
    )
)

In [ ]:
spot_balances_before = spot_balances_df.count()

spot_balances_bronze_df = (
    spot_balances_df
    .dropDuplicates([
        "asset",
        "_source_file"
    ])
)

spot_balances_after = spot_balances_bronze_df.count()
total_duplicates_removed = spot_balances_before - spot_balances_after

print("Rows before de-duplication:", spot_balances_before)
print("Rows after de-duplication:", spot_balances_after)
print("Total duplicates removed:", total_duplicates_removed)

In [ ]:
(
    spot_balances_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.giottus_spot_balances")
)

####Giottus Spot trades

In [ ]:
spot_trades_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("id", LongType(), True),
    StructField("orderId", StringType(), True),
    StructField("price", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("quoteQty", StringType(), True),
    StructField("commission", StringType(), True),
    StructField("commissionAsset", StringType(), True),
    StructField("tds", StringType(), True),
    StructField("time", LongType(), True),
    StructField("isBuyer", BooleanType(), True),
    StructField("isMaker", BooleanType(), True)
])

In [ ]:
spot_trades_raw_df = (
    spark.read
    .schema(spot_trades_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{base_path}/trades")
)

spot_trades_df = (
    spot_trades_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Giottus"))
    .withColumn("source_type", lit("Spot_Trade"))
    .withColumn("ingested_at", current_timestamp())
)

print("Spot trades rows:", spot_trades_df.count())

display(
    spot_trades_df.orderBy(
        col("time").desc()
    )
)

In [ ]:
spot_trades_rows_before = spot_trades_df.count()

spot_trades_bronze_df = (
    spot_trades_df
    .dropDuplicates([
        "symbol",
        "id"
    ])
)

spot_trades_rows_after = spot_trades_bronze_df.count()
total_duplicates_removed = spot_trades_rows_before - spot_trades_rows_after

print("Rows before de-duplication:", spot_trades_rows_before)
print("Rows after de-duplication:", spot_trades_rows_after)
print("Total duplicates removed:", total_duplicates_removed)

In [ ]:
(
    spot_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.giottus_spot_trades")
)

####Giottus EBES trades

In [ ]:
ebes_trades_schema = StructType([
    StructField("commission", StringType(), True),
    StructField("commissionAsset", StringType(), True),
    StructField("id", LongType(), True),
    StructField("isBuyer", BooleanType(), True),
    StructField("isMaker", BooleanType(), True),
    StructField("price", StringType(), True),
    StructField("qty", StringType(), True),
    StructField("quoteQty", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("tds", StringType(), True),
    StructField("time", LongType(), True)
])

In [ ]:
ebes_trades_raw_df = (
    spark.read
    .schema(ebes_trades_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{base_path}/ebes_trades/")
)

ebes_trades_df = (
    ebes_trades_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Giottus"))
    .withColumn("source_type", lit("Ebes_Trade"))
    .withColumn("ingested_at", current_timestamp())
)

print("Ebes trades rows:", ebes_trades_df.count())

display(
    ebes_trades_df.orderBy(
        col("time").desc()
    )
)

In [ ]:
ebes_trades_rows_before = ebes_trades_df.count()

ebes_trades_bronze_df = (
    ebes_trades_df
    .dropDuplicates([
        "symbol",
        "id"
    ])
)

ebes_trades_rows_after = ebes_trades_bronze_df.count()
total_duplicates_removed = ebes_trades_rows_before - ebes_trades_rows_after

print("Rows before de-duplication:", ebes_trades_rows_before)
print("Rows after de-duplication:", ebes_trades_rows_after)
print("Total duplicates removed:", total_duplicates_removed)

In [ ]:
(
    ebes_trades_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.giottus_ebes_trades")
)

####Giottus Crypto deposits

In [ ]:
crypto_deposits_schema = StructType([
    StructField("action", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("cryptoAmount", StringType(), True),
    StructField("status", StringType(), True),
    StructField("transactionHash", StringType(), True),
    StructField("time", LongType(), True),
    StructField("network", StringType(), True),
    StructField("fromAddress", StringType(), True),
    StructField("toAddress", StringType(), True),
    StructField("fee", StringType(), True)
])

In [ ]:
crypto_deposits_raw_df = (
    spark.read
    .schema(crypto_deposits_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{base_path}/crypto_deposits/")
)

crypto_deposits_df = (
    crypto_deposits_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Giottus"))
    .withColumn("source_type", lit("Crypto_Deposit"))
    .withColumn("ingested_at", current_timestamp())
)

print("Crypto deposits rows:", crypto_deposits_df.count())

display(
    crypto_deposits_df.orderBy(
        col("time").desc()
    )
)

In [ ]:
crypto_deposits_rows_before = crypto_deposits_df.count()

crypto_deposits_bronze_df = (
    crypto_deposits_df
    .dropDuplicates([
        "asset",
        "transactionHash"
    ])
)

crypto_deposits_rows_after = crypto_deposits_bronze_df.count()
total_duplicates_removed = crypto_deposits_rows_before - crypto_deposits_rows_after

print("Rows before de-duplication:", crypto_deposits_rows_before)
print("Rows after de-duplication:", crypto_deposits_rows_after)
print("Total duplicates removed:", total_duplicates_removed)

In [ ]:
(
    crypto_deposits_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.giottus_crypto_deposits")
)

####Crypto withdrawals

In [ ]:
crypto_withdrawals_schema = StructType([
    StructField("action", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("cryptoAmount", StringType(), True),
    StructField("status", StringType(), True),
    StructField("transactionHash", StringType(), True),
    StructField("time", LongType(), True),
    StructField("network", StringType(), True),
    StructField("fromAddress", StringType(), True),
    StructField("toAddress", StringType(), True),
    StructField("fee", StringType(), True)
])

In [ ]:
crypto_withdrawals_raw_df = (
    spark.read
    .schema(crypto_withdrawals_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{base_path}/crypto_withdrawals/")
)

crypto_withdrawals_df = (
    crypto_withdrawals_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Giottus"))
    .withColumn("source_type", lit("Crypto_Withdrawal"))
    .withColumn("ingested_at", current_timestamp())
)

print("Crypto withdrawals rows:", crypto_withdrawals_df.count())

display(
    crypto_withdrawals_df.orderBy(
        col("time").desc()
    )
)

In [ ]:
crypto_withdrawals_before = crypto_withdrawals_df.count()

crypto_withdrawals_bronze_df = (
    crypto_withdrawals_df
    .dropDuplicates([
        "asset",
        "transactionHash"
    ])
)

crypto_withdrawals_after = crypto_withdrawals_bronze_df.count()
total_duplicates_removed = crypto_withdrawals_before - crypto_withdrawals_after

print("Rows before de-duplication:", crypto_withdrawals_before)
print("Rows after de-duplication:", crypto_withdrawals_after)
print("Total duplicates removed:", total_duplicates_removed)


In [ ]:
(
    crypto_withdrawals_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.giottus_crypto_withdrawals")
)

####Giottus fiat deposits 

In [ ]:
fiat_deposits_schema = StructType([
    StructField("action", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("fee", StringType(), True),
    StructField("status", StringType(), True),
    StructField("referenceId", StringType(), True),
    StructField("time", LongType(), True)
])

In [ ]:
fiat_deposits_raw_df = (
    spark.read
    .schema(fiat_deposits_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{base_path}/fiat_deposits/")
)

fiat_deposits_df = (
    fiat_deposits_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Giottus"))
    .withColumn("source_type", lit("Fiat_Deposit"))
    .withColumn("ingested_at", current_timestamp())
)

print("Fiat deposits rows:", fiat_deposits_df.count())

display(
    fiat_deposits_df.orderBy(
        col("time").desc()
    )
)

In [ ]:
fiat_deposits_rows = fiat_deposits_df.count()

fiat_deposits_bronze_df = (
    fiat_deposits_df
    .dropDuplicates([
        "referenceId"
    ])
)

fiat_deposits_after = fiat_deposits_bronze_df.count()
total_duplicates_removed = fiat_deposits_rows - fiat_deposits_after

print("Rows before de-duplication:", fiat_deposits_rows)
print("Rows after de-duplication:", fiat_deposits_after)
print("Total duplicates removed:", total_duplicates_removed)

In [ ]:
(
    fiat_deposits_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.giottus_fiat_deposits")
)

####Giottus fiat withdrawals

In [ ]:
fiat_withdrawals_schema = StructType([
    StructField("action", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("fee", StringType(), True),
    StructField("status", StringType(), True),
    StructField("referenceId", StringType(), True),
    StructField("time", LongType(), True)
])

In [ ]:
fiat_withdrawals_raw_df = (
    spark.read
    .schema(fiat_withdrawals_schema)
    .option("multiLine", True)
    .option("recursiveFileLookup", True)
    .json(f"{base_path}/fiat_withdrawals/")
)

fiat_withdrawals_df = (
    fiat_withdrawals_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("Giottus"))
    .withColumn("source_type", lit("Fiat_Withdrawal"))
    .withColumn("ingested_at", current_timestamp())
)

print("Fiat withdrawals rows:", fiat_withdrawals_df.count())

display(
    fiat_withdrawals_df.orderBy(
        col("time").desc()
    )
)

In [ ]:
fiat_withdrawals_before = fiat_withdrawals_df.count()

fiat_withdrawals_bronze_df = (
    fiat_withdrawals_df
    .dropDuplicates([
        "referenceId"
    ])
)

fiat_withdrawals_after = fiat_withdrawals_bronze_df.count()
total_duplicates_removed = fiat_withdrawals_before - fiat_withdrawals_after

print("Rows before de-duplication:", fiat_withdrawals_before)
print("Rows after de-duplication:", fiat_withdrawals_after)
print("Total duplicates removed:", total_duplicates_removed)

In [ ]:
(
    fiat_withdrawals_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.giottus_fiat_withdrawals")
)